In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model, optimizers
import numpy as np
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/new approach v2/"

# PRE-TRAINING DATA (FULL DATASET)
X_VIEW1_PATH = os.path.join(BASE_PATH, "PRETRAIN_X_view1.npy")
X_VIEW2_PATH = os.path.join(BASE_PATH, "PRETRAIN_X_view2.npy")

# Output Weights
WEIGHTS_PATH = os.path.join(BASE_PATH, "FULL_HMVCL_Encoder_CNN2.weights.h5")

# Hyperparameters
BATCH_SIZE = 128
EPOCHS = 50
LEARNING_RATE = 0.0001
TEMPERATURE = 0.1
PROJECTION_DIM = 64
LATENT_DIM = 128

# --- 1. Data Loading & Cleaning ---
def load_data():
    print("--- 1. Loading Data ---")
    if not os.path.exists(X_VIEW1_PATH):
        print(f"FATAL: File not found: {X_VIEW1_PATH}")
        return None, None

    X_view1 = np.load(X_VIEW1_PATH).astype('float32')
    X_view2 = np.load(X_VIEW2_PATH).astype('float32')

    print(f"Original Shapes -> View 1: {X_view1.shape}, View 2: {X_view2.shape}")

    # CLEAN DIRTY DATA (NaN/Inf)
    print("--- Cleaning Data (NaN/Inf check) ---")
    if np.isnan(X_view2).any() or np.isinf(X_view2).any():
        print("   ! Found NaN/Inf in View 2. Replacing with zeros...")
        X_view2 = np.nan_to_num(X_view2, nan=0.0, posinf=0.0, neginf=0.0)

    # NORMALIZE STATISTICS
    print("--- Normalizing View 2 (Statistics) ---")
    scaler = StandardScaler()
    X_view2 = scaler.fit_transform(X_view2)

    print(f"   View 2 Statistics: Mean={np.mean(X_view2):.4f}, Std={np.std(X_view2):.4f}")

    return X_view1, X_view2

# --- 2. Architecture Definitions ---
def get_cnn_encoder(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = layers.Reshape((input_shape[0] * input_shape[1], 1))(inputs)
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Flatten()(x)
    h = layers.Dense(LATENT_DIM, activation='relu')(x)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    z = layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1, epsilon=1e-10))(z)
    return Model(inputs, [h, z], name="CNN_Encoder")

def get_mlp_encoder(input_dim):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Dense(256, activation='relu')(inputs)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    h = layers.Dense(LATENT_DIM, activation='relu')(x)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    z = layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1, epsilon=1e-10))(z)
    return Model(inputs, [h, z], name="MLP_Encoder")

# --- 3. Contrastive Model Wrapper ---
class HMVCL(Model):
    def __init__(self, cnn_encoder, mlp_encoder, temperature=0.1):
        super(HMVCL, self).__init__()
        self.cnn_encoder = cnn_encoder
        self.mlp_encoder = mlp_encoder
        self.temperature = temperature

    def compile(self, optimizer, loss_fn):
        super(HMVCL, self).compile()
        self.optimizer = optimizer
        self.loss_fn = loss_fn

    def train_step(self, data):
        # Unpacking is now safe because tf.data.Dataset guarantees structure
        view1, view2 = data

        with tf.GradientTape() as tape:
            _, z1 = self.cnn_encoder(view1, training=True)
            _, z2 = self.mlp_encoder(view2, training=True)
            loss = self.loss_fn(z1, z2, self.temperature)

        gradients = tape.gradient(loss, self.cnn_encoder.trainable_variables + self.mlp_encoder.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.cnn_encoder.trainable_variables + self.mlp_encoder.trainable_variables))
        return {"loss": loss}

# --- 4. Loss Function ---
def nt_xent_loss(z1, z2, temperature):
    batch_size = tf.shape(z1)[0]
    z = tf.concat([z1, z2], axis=0)
    sim_matrix = tf.matmul(z, z, transpose_b=True)
    sim_matrix = sim_matrix / temperature

    mask = tf.eye(2 * batch_size, dtype=tf.bool)
    sim_matrix = tf.where(mask, -1e9, sim_matrix)

    labels = tf.concat([tf.range(batch_size) + batch_size, tf.range(batch_size)], axis=0)
    loss = tf.keras.losses.sparse_categorical_crossentropy(labels, sim_matrix, from_logits=True)
    return tf.reduce_mean(loss)

# --- 5. Main ---
def main():
    X1, X2 = load_data()
    if X1 is None: return

    # --- FIX: Create tf.data.Dataset ---
    # This ensures 'train_step' receives exactly (view1, view2)
    print("--- Creating tf.data.Dataset ---")
    dataset = tf.data.Dataset.from_tensor_slices((X1, X2))
    dataset = dataset.shuffle(buffer_size=1024).batch(BATCH_SIZE)

    # Build Models
    cnn = get_cnn_encoder((10, 784))
    mlp = get_mlp_encoder(X2.shape[1])

    hmvcl = HMVCL(cnn, mlp, temperature=TEMPERATURE)
    opt = optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=1.0)
    hmvcl.compile(optimizer=opt, loss_fn=nt_xent_loss)

    print("\n--- Starting Contrastive Pre-Training (Fixed) ---")
    # Pass the dataset, not the raw lists
    hmvcl.fit(
        dataset,
        epochs=EPOCHS,
        verbose=1
    )

    print(f"Saving weights to {WEIGHTS_PATH}")
    hmvcl.cnn_encoder.save_weights(WEIGHTS_PATH)
    print("Done.")

if __name__ == "__main__":
    main()